In [ ]:
import os
import re
import subprocess
import sys

def pip(packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])

if "jax" in sys.modules:
    raise RuntimeError("Restart session, then run this cell first before importing jax.")

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ.pop("JAX_PLATFORMS", None)
os.environ.pop("PJRT_DEVICE", None)
os.environ.pop("CUDA_VISIBLE_DEVICES", None)

jax_extra = os.environ.get("JAX_CUDA_EXTRA", "cuda12")

out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=30)
smi = (out.stdout or "") + (out.stderr or "")
print("--- nvidia-smi -L ---")
print(smi)
print("returncode:", out.returncode)

has_gpu = re.search(r"GPU\s+\d+:", smi) is not None
bad = "libnvidia-ml" in smi.lower() or "couldn't find libnvidia" in smi.lower()
if not has_gpu and (bad or out.returncode != 0):
    raise RuntimeError("No GPU visible to this kernel. Runtime → Change runtime type → GPU.")

pip(["--upgrade", "pip"])
pip(["--upgrade", f"jax[{jax_extra}]", "optax", "torch", "torchvision", "torchaudio"])

print("Install done. Now go to Runtime → Restart session, then run the next cell.")

--- nvidia-smi -L ---
GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-37044b8d-4ac1-6d33-d22f-87ecd5945fee)

returncode: 0
Install done. Now go to Runtime → Restart session, then run the next cell.


In [ ]:
import torch
import jax
import jax.numpy as jnp

print("torch", torch.__version__, "cuda", torch.cuda.is_available())
print("JAX", jax.__version__, jax.devices(), jax.default_backend())

x = jnp.ones((256, 256), jnp.float32)
jax.block_until_ready(x @ x)
print("JAX GPU OK")

torch 2.11.0+cu130 cuda True
JAX 0.9.2 [CudaDevice(id=0)] gpu
JAX GPU OK


In [ ]:
import pathlib
import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
path = pathlib.Path("input.txt")
if not path.exists():
    urllib.request.urlretrieve(url, path)
print(path.resolve(), path.stat().st_size)

/content/input.txt 1115394


In [ ]:
%%writefile model.py
"""
Shared, parameterized GPT architecture for the research experiments.


This is the same decoder-only transformer used in all hand-designed models
(and in v2.py), but with hyperparameters passed as constructor arguments
instead of module-level globals."""


import torch
import torch.nn as nn
from torch.nn import functional as F


class Head(nn.Module):
    """One head of self-attention."""

    def __init__(self, n_embd, head_size, block_size, dropout=0.0):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v


class MultiHeadAttention(nn.Module):
    """Multiple heads of self-attention in parallel."""

    def __init__(self, n_embd, num_heads, head_size, block_size, dropout=0.0):
        super().__init__()
        self.heads = nn.ModuleList(
            [Head(n_embd, head_size, block_size, dropout) for _ in range(num_heads)]
        )
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    """Simple linear → ReLU → linear → dropout."""

    def __init__(self, n_embd, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Transformer block: LayerNorm → Attention → Residual → LayerNorm → FFN → Residual."""

    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_embd, n_head, head_size, block_size, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class GPT(nn.Module):
    """
    Tiny decoder-only GPT.

    Args:
        vocab_size: number of tokens
        n_embd:     embedding dimension
        n_head:     number of attention heads
        n_layer:    number of transformer blocks
        block_size: maximum sequence length
        dropout:    dropout rate (default 0.0)
    """

    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout=0.0):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=True)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        device = idx.device
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        if targets is None:
            return logits, None

        B, T, C = logits.shape
        logits_flat = logits.view(B * T, C)
        targets_flat = targets.view(B * T)
        loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=-100)
        return logits, loss

    def get_attention_weights(self, idx):
        """
        Run a forward pass and return the attention weight matrices
        from every head in every layer.

        Returns:
            list of list of tensors: weights[layer][head] is (B, T, T)
        """
        B, T = idx.shape
        device = idx.device
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb

        all_weights = []
        for block in self.blocks:
            x_ln = block.ln1(x)
            layer_weights = []
            head_outputs = []
            for head in block.sa.heads:
                k = head.key(x_ln)
                q = head.query(x_ln)
                wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
                wei = wei.masked_fill(head.tril[:T, :T] == 0, float('-inf'))
                wei = F.softmax(wei, dim=-1)
                layer_weights.append(wei.detach())
                v = head.value(x_ln)
                head_outputs.append(wei @ v)
            all_weights.append(layer_weights)
            attn_out = torch.cat(head_outputs, dim=-1)
            attn_out = block.sa.dropout(block.sa.proj(attn_out))
            x = x + attn_out
            x = x + block.ffwd(block.ln2(x))

        return all_weights


Writing model.py


In [ ]:
%%writefile jax_lm.py
"""
JAX implementation of the same decoder-only GPT as `model.py` (research).

Used for timing / throughput comparisons against PyTorch. Not wired into the
generalization experiments (those stay PyTorch + `model.py`).

Dependencies: pip install jax optax
  - GPU: follow https://github.com/jax-ml/jax (e.g. jax[cuda12] on Linux CUDA)
  - Apple Silicon: JAX often runs on CPU; see JAX install docs for experimental GPU.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Tuple

import jax
import jax.numpy as jnp
import numpy as np
import optax


@dataclass(frozen=True)
class GPTConfig:
    vocab_size: int
    n_embd: int
    n_head: int
    n_layer: int
    block_size: int
    dropout: float = 0.0


def layer_norm(x: jnp.ndarray, scale: jnp.ndarray, bias: jnp.ndarray, eps: float = 1e-5) -> jnp.ndarray:
    mean = jnp.mean(x, axis=-1, keepdims=True)
    var = jnp.var(x, axis=-1, keepdims=True)
    x_norm = (x - mean) / jnp.sqrt(var + eps)
    return scale * x_norm + bias


def dropout(rng: jax.Array, x: jnp.ndarray, rate: float, train: bool) -> jnp.ndarray:
    if not train or rate <= 0.0:
        return x
    keep = 1.0 - rate
    mask = jax.random.bernoulli(rng, p=keep, shape=x.shape)
    return jnp.where(mask, x / keep, 0.0)


def linear(x: jnp.ndarray, kernel: jnp.ndarray, bias: jnp.ndarray | None) -> jnp.ndarray:
    # kernel: (in_dim, out_dim)  ->  x @ kernel + bias
    y = jnp.dot(x, kernel)
    if bias is not None:
        y = y + bias
    return y


def causal_mask(T: int) -> jnp.ndarray:
    return jnp.tril(jnp.ones((T, T), dtype=jnp.float32))


def head_forward(
    rng: jax.Array,
    x: jnp.ndarray,
    Wq: jnp.ndarray,
    Wk: jnp.ndarray,
    Wv: jnp.ndarray,
    tril_T: jnp.ndarray,
    dropout_rate: float,
    train: bool,
) -> jnp.ndarray:
    """Single attention head. Shapes: W* are (n_embd, head_size). tril_T is (T, T)."""
    hs = Wq.shape[1]
    q = jnp.dot(x, Wq)
    k = jnp.dot(x, Wk)
    v = jnp.dot(x, Wv)
    wei = jnp.matmul(q, jnp.swapaxes(k, -2, -1)) * (hs ** -0.5)
    wei = jnp.where(tril_T[None, :, :] == 0, -1e9, wei)
    wei = jax.nn.softmax(wei, axis=-1)
    rng, dr = jax.random.split(rng)
    wei = dropout(dr, wei, dropout_rate, train)
    return jnp.matmul(wei, v)


def block_forward(
    rng: jax.Array,
    x: jnp.ndarray,
    block: Dict[str, Any],
    tril_T: jnp.ndarray,
    n_head: int,
    dropout_rate: float,
    train: bool,
) -> jnp.ndarray:
    ln1s, ln1b = block["ln1_scale"], block["ln1_bias"]
    ln2s, ln2b = block["ln2_scale"], block["ln2_bias"]

    h = layer_norm(x, ln1s, ln1b)
    head_outs = []
    for i in range(n_head):
        rng, hr = jax.random.split(rng)
        ho = head_forward(
            hr,
            h,
            block["Wq"][i],
            block["Wk"][i],
            block["Wv"][i],
            tril_T,
            dropout_rate,
            train,
        )
        head_outs.append(ho)
    sa = jnp.concatenate(head_outs, axis=-1)
    rng, dr = jax.random.split(rng)
    sa = linear(sa, block["proj_kernel"], block["proj_bias"])
    sa = dropout(dr, sa, dropout_rate, train)
    x = x + sa

    h2 = layer_norm(x, ln2s, ln2b)
    rng, dr = jax.random.split(rng)
    ff = linear(h2, block["ff1_kernel"], block["ff1_bias"])
    ff = jax.nn.relu(ff)
    ff = linear(ff, block["ff2_kernel"], block["ff2_bias"])
    ff = dropout(dr, ff, dropout_rate, train)
    x = x + ff
    return x


def gpt_forward(
    rng: jax.Array,
    params: Dict[str, Any],
    idx: jnp.ndarray,
    train: bool,
    cfg: GPTConfig,
) -> jnp.ndarray:
    """Returns logits (B, T, vocab). `params` is array-only (no config nested)."""
    B, T = idx.shape
    te = params["wte"][idx]
    pos = jnp.arange(T)[None, :]
    pe = params["wpe"][pos]
    x = te + pe
    tril_T = causal_mask(T)
    for i in range(cfg.n_layer):
        rng, lr = jax.random.split(rng)
        x = block_forward(lr, x, params["blocks"][i], tril_T, cfg.n_head, cfg.dropout, train)
    x = layer_norm(x, params["lnf_scale"], params["lnf_bias"])
    logits = linear(x, params["lm_head_kernel"], params["lm_head_bias"])
    return logits


def loss_fn(
    params: Dict[str, Any],
    rng: jax.Array,
    idx: jnp.ndarray,
    targets: jnp.ndarray,
    train: bool,
    cfg: GPTConfig,
) -> jnp.ndarray:
    logits = gpt_forward(rng, params, idx, train=train, cfg=cfg)
    log_probs = jax.nn.log_softmax(logits, axis=-1)
    mask = targets != -100
    safe_targets = jnp.where(mask, targets, 0)
    token_loss = -jnp.take_along_axis(log_probs, safe_targets[..., None], axis=-1).squeeze(-1)
    return jnp.sum(token_loss * mask) / jnp.maximum(jnp.sum(mask), 1.0)


def init_params(key: jax.Array, cfg: GPTConfig) -> Dict[str, Any]:
    """Initialize parameters (similar scale to default PyTorch nn.Module init)."""
    keys = jax.random.split(key, 256)
    ki = 0
    scale = 0.02

    def randn(shape):
        nonlocal ki
        k = keys[ki]
        ki += 1
        return jax.random.normal(k, shape, dtype=jnp.float32) * scale

    d = cfg.n_embd
    hs = d // cfg.n_head
    T = cfg.block_size
    V = cfg.vocab_size

    wte = randn((V, d))
    wpe = randn((T, d))

    blocks = []
    for _ in range(cfg.n_layer):
        Wq = jnp.stack([randn((d, hs)) for _ in range(cfg.n_head)], axis=0)
        Wk = jnp.stack([randn((d, hs)) for _ in range(cfg.n_head)], axis=0)
        Wv = jnp.stack([randn((d, hs)) for _ in range(cfg.n_head)], axis=0)
        proj_kernel = randn((d, d))
        proj_bias = jnp.zeros((d,), dtype=jnp.float32)
        ff1_kernel = randn((d, 4 * d))
        ff1_bias = jnp.zeros((4 * d,), dtype=jnp.float32)
        ff2_kernel = randn((4 * d, d))
        ff2_bias = jnp.zeros((d,), dtype=jnp.float32)
        ln1_scale = jnp.ones((d,), dtype=jnp.float32)
        ln1_bias = jnp.zeros((d,), dtype=jnp.float32)
        ln2_scale = jnp.ones((d,), dtype=jnp.float32)
        ln2_bias = jnp.zeros((d,), dtype=jnp.float32)
        blocks.append(
            {
                "Wq": Wq,
                "Wk": Wk,
                "Wv": Wv,
                "proj_kernel": proj_kernel,
                "proj_bias": proj_bias,
                "ff1_kernel": ff1_kernel,
                "ff1_bias": ff1_bias,
                "ff2_kernel": ff2_kernel,
                "ff2_bias": ff2_bias,
                "ln1_scale": ln1_scale,
                "ln1_bias": ln1_bias,
                "ln2_scale": ln2_scale,
                "ln2_bias": ln2_bias,
            }
        )

    lnf_scale = jnp.ones((d,), dtype=jnp.float32)
    lnf_bias = jnp.zeros((d,), dtype=jnp.float32)
    lm_head_kernel = randn((d, V))
    lm_head_bias = jnp.zeros((V,), dtype=jnp.float32)

    return {
        "wte": wte,
        "wpe": wpe,
        "blocks": tuple(blocks),
        "lnf_scale": lnf_scale,
        "lnf_bias": lnf_bias,
        "lm_head_kernel": lm_head_kernel,
        "lm_head_bias": lm_head_bias,
    }


def build_optimizer(learning_rate: float, weight_decay: float = 0.0):
    return optax.adamw(learning_rate, b1=0.9, b2=0.999, eps=1e-8, weight_decay=weight_decay)


def train_step_factory(cfg: GPTConfig, learning_rate: float):
    optimizer = build_optimizer(learning_rate)

    def loss_train(p, r, xi, yi):
        return loss_fn(p, r, xi, yi, train=True, cfg=cfg)

    @jax.jit
    def _step(params, opt_state, rng, x, y):
        loss, grads = jax.value_and_grad(loss_train)(params, rng, x, y)
        updates, new_opt = optimizer.update(grads, opt_state, params)
        new_params = optax.apply_updates(params, updates)
        return new_params, new_opt, loss

    return _step, optimizer


def numpy_batches(
    data: np.ndarray,
    batch_size: int,
    block_size: int,
    n_batches: int,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray]:
    """Precompute random batches shared by Torch/JAX for fair timing."""
    rng = np.random.RandomState(seed)
    max_i = len(data) - block_size
    xs = np.zeros((n_batches, batch_size, block_size), dtype=np.int64)
    ys = np.zeros((n_batches, batch_size, block_size), dtype=np.int64)
    for b in range(n_batches):
        ix = rng.randint(0, max_i, size=(batch_size,))
        for j, i in enumerate(ix):
            xs[b, j] = data[i : i + block_size]
            ys[b, j] = data[i + 1 : i + block_size + 1]
    return xs, ys


Writing jax_lm.py


In [ ]:
# --- imports (needed after %%writefile) ---
import math
import os
import sys
import time
from pathlib import Path

import numpy as np
import torch

sys.path.insert(0, str(Path.cwd()))
from model import GPT
from jax_lm import GPTConfig, init_params, loss_fn, train_step_factory

# --- config ---
HARDCORE = True
MAX_ITERS = 5000
EVAL_INTERVAL = 500
EVAL_ITERS = 200
BLOCK_SIZE = 256
DROPOUT = 0.2
LR = 3e-3
SEED = 1337
RUN_TORCH = True   # set False if you only want JAX
RUN_JAX = True

if HARDCORE:
    N_LAYER, N_HEAD, N_EMBD = 8, 8, 512
    BATCH_SIZE = 64
else:
    N_LAYER, N_HEAD, N_EMBD = 6, 6, 384
    BATCH_SIZE = 32

print(
    f"Preset: {'HARDCORE (8L/8H/512, b64)' if HARDCORE else 'v2.py (6L/6H/384, b32)'} | "
    f"max_iters={MAX_ITERS} eval={EVAL_INTERVAL}/{EVAL_ITERS}"
)

def resolve_input_path():
    cwd = Path.cwd().resolve()
    for p in (
        cwd / "input.txt",
        cwd.parent / "input.txt",
    ):
        if p.is_file():
            return p.resolve()
    return (cwd / "input.txt").resolve()

def load_encode(path: Path):
    text = path.read_text(encoding="utf-8")
    chars = sorted(list(set(text)))
    stoi = {ch: i for i, ch in enumerate(chars)}
    data = np.array([stoi[c] for c in text], dtype=np.int64)
    return data, len(chars)

def make_batches(data, n_batches, batch_size, block_size, seed):
    rng = np.random.RandomState(seed)
    max_i = len(data) - block_size
    xs = np.zeros((n_batches, batch_size, block_size), dtype=np.int64)
    ys = np.zeros((n_batches, batch_size, block_size), dtype=np.int64)
    for b in range(n_batches):
        ix = rng.randint(0, max_i, size=(batch_size,))
        for j, i in enumerate(ix):
            xs[b, j] = data[i : i + block_size]
            ys[b, j] = data[i + 1 : i + block_size + 1]
    return xs, ys

def count_torch_params(model):
    return sum(p.numel() for p in model.parameters())

def count_jax_params(params):
    import jax.tree_util as jtu
    return int(sum(int(x.size) for x in jtu.tree_leaves(params) if hasattr(x, "size")))

def sanity_check_devices():
    import jax
    print("--- device checks ---")
    print("PyTorch CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
    print("JAX:", jax.devices())
    print("---------------------")

@torch.no_grad()
def torch_estimate_loss(model, train_x, train_y, val_x, val_y, device, eval_iters):
    model.eval()
    tl = torch.zeros(eval_iters, device=device)
    vl = torch.zeros(eval_iters, device=device)
    for k in range(eval_iters):
        x = torch.from_numpy(train_x[k]).to(device=device, dtype=torch.long)
        y = torch.from_numpy(train_y[k]).to(device=device, dtype=torch.long)
        _, loss = model(x, y)
        tl[k] = loss
    for k in range(eval_iters):
        x = torch.from_numpy(val_x[k]).to(device=device, dtype=torch.long)
        y = torch.from_numpy(val_y[k]).to(device=device, dtype=torch.long)
        _, loss = model(x, y)
        vl[k] = loss
    model.train()
    return float(tl.mean().item()), float(vl.mean().item())

def run_pytorch(train_xs, train_ys, train_eval_x, train_eval_y, val_eval_x, val_eval_y, vocab_size, T, seed):
    device = (
        torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    )
    torch.manual_seed(seed)
    model = GPT(vocab_size, N_EMBD, N_HEAD, N_LAYER, T, DROPOUT).to(device)
    print(f"  PyTorch parameters: {count_torch_params(model):,}")
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    t0 = time.perf_counter()
    last_train, last_val = float("nan"), float("nan")
    for it in range(MAX_ITERS):
        x = torch.from_numpy(train_xs[it]).to(device=device, dtype=torch.long)
        y = torch.from_numpy(train_ys[it]).to(device=device, dtype=torch.long)
        opt.zero_grad(set_to_none=True)
        _, loss = model(x, y)
        if it == 0 and not torch.isfinite(loss).all():
            raise RuntimeError("NaN/Inf loss step 0 (PyTorch)")
        loss.backward()
        opt.step()
        if it % EVAL_INTERVAL == 0 or it == MAX_ITERS - 1:
            tr, va = torch_estimate_loss(model, train_eval_x, train_eval_y, val_eval_x, val_eval_y, device, EVAL_ITERS)
            last_train, last_val = tr, va
            print(f"  [pt] step {it:5d} | train {tr:.4f} | val {va:.4f} | elapsed {time.perf_counter()-t0:.1f}s", flush=True)
            if device.type == "cuda":
                torch.cuda.synchronize()
    wall = time.perf_counter() - t0
    tok = MAX_ITERS * BATCH_SIZE * BLOCK_SIZE
    print(f"  device={device}  wall_s={wall:.1f}s  ~tok/s {tok/wall:.0f}")
    print(f"  final train/val={last_train:.4f} / {last_val:.4f}")

def run_jax(train_xs, train_ys, train_eval_x, train_eval_y, val_eval_x, val_eval_y, vocab_size, T, seed):
    import jax
    import jax.numpy as jnp

    key = jax.random.PRNGKey(seed)
    cfg = GPTConfig(
        vocab_size=vocab_size,
        n_embd=N_EMBD,
        n_head=N_HEAD,
        n_layer=N_LAYER,
        block_size=T,
        dropout=DROPOUT,
    )
    key, k_init = jax.random.split(key)
    params = init_params(k_init, cfg)
    print(f"  JAX parameters: {count_jax_params(params):,}")
    step, tx = train_step_factory(cfg, LR)
    opt_state = tx.init(params)

    @jax.jit
    def eval_batch_loss(p, rng, xb, yb):
        return loss_fn(p, rng, xb, yb, train=False, cfg=cfg)

    def jax_mean_eval(p, key, x_stack, y_stack):
        s = 0.0
        k = key
        for i in range(EVAL_ITERS):
            k, sk = jax.random.split(k)
            xi = jnp.asarray(x_stack[i], dtype=jnp.int32)
            yi = jnp.asarray(y_stack[i], dtype=jnp.int32)
            li = eval_batch_loss(p, sk, xi, yi)
            jax.block_until_ready(li)
            s += float(li)
        return s / EVAL_ITERS

    t0 = time.perf_counter()
    last_train, last_val = float("nan"), float("nan")
    key_loop = key
    for it in range(MAX_ITERS):
        x = jnp.asarray(train_xs[it], dtype=jnp.int32)
        y = jnp.asarray(train_ys[it], dtype=jnp.int32)
        key_loop, k_step = jax.random.split(key_loop)
        params, opt_state, loss_t = step(params, opt_state, k_step, x, y)
        if it == 0:
            jax.block_until_ready(loss_t)
            if not math.isfinite(float(loss_t)):
                raise RuntimeError("NaN/Inf loss step 0 (JAX)")
        if it % EVAL_INTERVAL == 0 or it == MAX_ITERS - 1:
            key_loop, ke = jax.random.split(key_loop)
            tr = jax_mean_eval(params, ke, train_eval_x, train_eval_y)
            key_loop, ke2 = jax.random.split(key_loop)
            va = jax_mean_eval(params, ke2, val_eval_x, val_eval_y)
            last_train, last_val = tr, va
            print(f"  [jx] step {it:5d} | train {last_train:.4f} | val {last_val:.4f} | elapsed {time.perf_counter()-t0:.1f}s", flush=True)
    wall = time.perf_counter() - t0
    tok = MAX_ITERS * BATCH_SIZE * BLOCK_SIZE
    print(f"  device={jax.devices()[0]}  wall_s={wall:.1f}s  ~tok/s {tok/wall:.0f}")
    print(f"  final train/val={last_train:.4f} / {last_val:.4f}")

# --- run ---
DATA_PATH = resolve_input_path()
sanity_check_devices()
if not DATA_PATH.exists():
    raise FileNotFoundError("Missing input.txt — run the download cell.")
print("Using:", DATA_PATH, DATA_PATH.stat().st_size)
data, vocab_size = load_encode(DATA_PATH)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f"vocab={vocab_size} train={n} val={len(val_data)}")
t0 = time.perf_counter()
train_xs, train_ys = make_batches(train_data, MAX_ITERS, BATCH_SIZE, BLOCK_SIZE, SEED)
train_eval_x, train_eval_y = make_batches(train_data, EVAL_ITERS, BATCH_SIZE, BLOCK_SIZE, SEED + 1)
val_eval_x, val_eval_y = make_batches(val_data, EVAL_ITERS, BATCH_SIZE, BLOCK_SIZE, SEED + 2)
print(f"precompute batches: {time.perf_counter()-t0:.2f}s")

if RUN_TORCH:
    print("\n[PyTorch]")
    run_pytorch(train_xs, train_ys, train_eval_x, train_eval_y, val_eval_x, val_eval_y, vocab_size, BLOCK_SIZE, SEED)
if RUN_JAX:
    print("\n[JAX]")
    run_jax(train_xs, train_ys, train_eval_x, train_eval_y, val_eval_x, val_eval_y, vocab_size, BLOCK_SIZE, SEED)
print("\nDone.")

Preset: HARDCORE (8L/8H/512, b64) | max_iters=5000 eval=500/200
--- device checks ---
PyTorch CUDA: True NVIDIA A100-SXM4-80GB
JAX: [CudaDevice(id=0)]
---------------------
Using: /content/input.txt 1115394
vocab=65 train=1003854 val=111540
precompute batches: 1.40s

[PyTorch]
  PyTorch parameters: 25,405,505
  [pt] step     0 | train 5.8863 | val 5.9515 | elapsed 28.4s
  [pt] step   500 | train 2.5234 | val 2.5287 | elapsed 168.2s
  [pt] step  1000 | train 2.5309 | val 2.5414 | elapsed 307.9s
  [pt] step  1500 | train 2.5793 | val 2.5844 | elapsed 447.5s
  [pt] step  2000 | train 2.6242 | val 2.6298 | elapsed 586.9s
  [pt] step  2500 | train 2.7135 | val 2.7239 | elapsed 726.4s
  [pt] step  3000 | train 2.6876 | val 2.6993 | elapsed 866.0s
  [pt] step  3500 | train 2.7362 | val 2.7430 | elapsed 1005.7s
  [pt] step  4000 | train 2.7210 | val 2.7394 | elapsed 1145.3s
  [pt] step  4500 | train 2.7296 | val 2.7448 | elapsed 1284.9s
  [pt] step  4999 | train 2.7432 | val 2.7551 | elapsed 1